# ShopSphere – E-commerce Orders and Customer Analytics
Run the generator and loader scripts first. Configure `.env` with your local MySQL credentials.

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append(str(Path.cwd().parent / 'python'))
from database_connection import get_connection
conn = get_connection()


## Monthly revenue

In [ ]:
monthly = pd.read_sql('SELECT * FROM v_monthly_revenue ORDER BY revenue_month', conn)
display(monthly)
monthly.plot(x='revenue_month', y='revenue', kind='line', marker='o', title='Monthly Revenue')
plt.xticks(rotation=45); plt.tight_layout(); plt.show()


## Revenue by category

In [ ]:
category = pd.read_sql('''SELECT c.name category, SUM(oi.quantity*oi.unit_price) revenue
FROM categories c JOIN products p USING(category_id)
JOIN order_items oi USING(product_id) JOIN orders o USING(order_id)
WHERE o.status <> 'Cancelled' GROUP BY c.category_id,c.name ORDER BY revenue DESC''', conn)
display(category)
category.plot(x='category', y='revenue', kind='bar', title='Revenue by Category', legend=False)
plt.tight_layout(); plt.show()


## Customer lifetime value

In [ ]:
ltv = pd.read_sql('''SELECT c.customer_id,c.name,
SUM(oi.quantity*oi.unit_price) lifetime_value
FROM customers c JOIN orders o USING(customer_id)
JOIN order_items oi USING(order_id) WHERE o.status <> 'Cancelled'
GROUP BY c.customer_id,c.name ORDER BY lifetime_value DESC LIMIT 10''', conn)
display(ltv)
conn.close()
